In [43]:
import random
from datetime import datetime, timedelta
import pandas as pd

In [44]:
# Set random seed for reproducibility
random.seed(42)

In [45]:
# Configuration parameters
NUM_PATIENTS = 200
start_date = datetime(2026, 6, 1, 8, 0, 0)

In [46]:
# Reference data pools
genders = ["Male", "Female", "Other"]
doctors = ["Dr. Robert", "Dr. House", "Dr. Smith", "Dr. Adams", "Dr. Chen"]
triage_levels = ["ESI Level 2", "ESI Level 3", "ESI Level 4", "ESI Level 5"]

In [47]:
# Locations mapped to activities
activity_locations = {
    "Check-in": "Reception Desk",
    "Triage": "Triage Room 1",
    "Doctor Consultation": "Consultation Bay B",
    "X-Ray": "Radiology Lab 2",
    "Discharge": "Exit Lobby",
}

In [48]:
resources = {
    "Check-in": "Front Desk Staff",
    "Triage": "Nurse Sarah",
    "Doctor Consultation": "Attending Physician",
    "X-Ray": "Tech Mike",
    "Discharge": "Discharge Coordinator",
}

In [49]:
# Reset event rows
event_rows = []

# Consolidated loop to generate all 200 sequential patient records (1000 to 1199)
for i in range(NUM_PATIENTS):
    case_id = 1000 + i
    patient_age = random.randint(18, 85)
    patient_gender = random.choice(genders)
    attending_doctor = random.choice(doctors)
    acuity_level = random.choice(triage_levels)

    # 40% chance that a patient is missing paperwork, triggering the rework loop after X-Ray
    paperwork_missing_flag = random.random() < 0.40

    # Stagger patient arrival times
    current_time = start_date + timedelta(
        minutes=random.randint(0, 1440 * 3)
    )  # Spread over 3 days

    # 1. Check-in
    event_rows.append({
        "case_id": case_id,
        "activity": "Check-in",
        "timestamp": current_time,
        "resource": resources["Check-in"],
        "location": activity_locations["Check-in"],
        "acuity_level": acuity_level,
        "patient_age": patient_age,
        "patient_gender": patient_gender,
        "attending_doctor": attending_doctor,
        "paperwork_missing_flag": paperwork_missing_flag,
    })

    # 2. Triage
    current_time += timedelta(minutes=random.randint(5, 20))
    event_rows.append({
        "case_id": case_id,
        "activity": "Triage",
        "timestamp": current_time,
        "resource": resources["Triage"],
        "location": activity_locations["Triage"],
        "acuity_level": acuity_level,
        "patient_age": patient_age,
        "patient_gender": patient_gender,
        "attending_doctor": attending_doctor,
        "paperwork_missing_flag": paperwork_missing_flag,
    })

    # 3. Doctor Consultation
    current_time += timedelta(minutes=random.randint(15, 60))
    event_rows.append({
        "case_id": case_id,
        "activity": "Doctor Consultation",
        "timestamp": current_time,
        "resource": attending_doctor,
        "location": activity_locations["Doctor Consultation"],
        "acuity_level": acuity_level,
        "patient_age": patient_age,
        "patient_gender": patient_gender,
        "attending_doctor": attending_doctor,
        "paperwork_missing_flag": paperwork_missing_flag,
    })

    # 4. X-Ray
    current_time += timedelta(minutes=random.randint(10, 30))
    event_rows.append({
        "case_id": case_id,
        "activity": "X-Ray",
        "timestamp": current_time,
        "resource": resources["X-Ray"],
        "location": activity_locations["X-Ray"],
        "acuity_level": acuity_level,
        "patient_age": patient_age,
        "patient_gender": patient_gender,
        "attending_doctor": attending_doctor,
        "paperwork_missing_flag": paperwork_missing_flag,
    })

    # --- INJECTED BOTTLENECK / REWORK LOOP ---
    if paperwork_missing_flag:
        # Sent back to Triage due to missing form
        current_time += timedelta(minutes=random.randint(15, 30))
        event_rows.append({
            "case_id": case_id,
            "activity": "Triage",  # Rework step
            "timestamp": current_time,
            "resource": resources["Triage"],
            "location": activity_locations["Triage"],
            "acuity_level": acuity_level,
            "patient_age": patient_age,
            "patient_gender": patient_gender,
            "attending_doctor": attending_doctor,
            "paperwork_missing_flag": True,
        })

        # Quick re-consultation after form fix
        current_time += timedelta(minutes=random.randint(10, 25))
        event_rows.append({
            "case_id": case_id,
            "activity": "Doctor Consultation",
            "timestamp": current_time,
            "resource": attending_doctor,
            "location": activity_locations["Doctor Consultation"],
            "acuity_level": acuity_level,
            "patient_age": patient_age,
            "patient_gender": patient_gender,
            "attending_doctor": attending_doctor,
            "paperwork_missing_flag": False,
        })

    # 5. Discharge
    current_time += timedelta(minutes=random.randint(15, 40))
    event_rows.append({
        "case_id": case_id,
        "activity": "Discharge",
        "timestamp": current_time,
        "resource": resources["Discharge"],
        "location": activity_locations["Discharge"],
        "acuity_level": acuity_level,
        "patient_age": patient_age,
        "patient_gender": patient_gender,
        "attending_doctor": attending_doctor,
        "paperwork_missing_flag": paperwork_missing_flag,
    })

In [50]:
# Convert to DataFrame and sort before printing
df_events = pd.DataFrame(event_rows)
df_events = df_events.sort_values(by=["case_id", "timestamp"]).reset_index(drop=True)

print(df_events['case_id'])

0       1000
1       1000
2       1000
3       1000
4       1000
        ... 
1137    1199
1138    1199
1139    1199
1140    1199
1141    1199
Name: case_id, Length: 1142, dtype: int64


In [51]:
# Preview the first 10 rows
print(df_events.head(10))

   case_id             activity           timestamp               resource  \
0     1000             Check-in 2026-06-01 21:59:00       Front Desk Staff   
1     1000               Triage 2026-06-01 22:06:00            Nurse Sarah   
2     1000  Doctor Consultation 2026-06-01 22:58:00              Dr. Smith   
3     1000                X-Ray 2026-06-01 23:21:00              Tech Mike   
4     1000               Triage 2026-06-01 23:37:00            Nurse Sarah   
5     1000  Doctor Consultation 2026-06-01 23:47:00              Dr. Smith   
6     1000            Discharge 2026-06-02 00:04:00  Discharge Coordinator   
7     1001             Check-in 2026-06-03 17:16:00       Front Desk Staff   
8     1001               Triage 2026-06-03 17:28:00            Nurse Sarah   
9     1001  Doctor Consultation 2026-06-03 18:11:00               Dr. Chen   

             location acuity_level  patient_age patient_gender  \
0      Reception Desk  ESI Level 3           32           Male   
1       T

In [52]:
# Preview the last 10 rows
print(df_events.tail(10))

      case_id             activity           timestamp               resource  \
1132     1198             Check-in 2026-06-02 05:33:00       Front Desk Staff   
1133     1198               Triage 2026-06-02 05:51:00            Nurse Sarah   
1134     1198  Doctor Consultation 2026-06-02 06:47:00              Dr. House   
1135     1198                X-Ray 2026-06-02 07:12:00              Tech Mike   
1136     1198            Discharge 2026-06-02 07:42:00  Discharge Coordinator   
1137     1199             Check-in 2026-06-01 20:12:00       Front Desk Staff   
1138     1199               Triage 2026-06-01 20:24:00            Nurse Sarah   
1139     1199  Doctor Consultation 2026-06-01 21:13:00              Dr. Smith   
1140     1199                X-Ray 2026-06-01 21:24:00              Tech Mike   
1141     1199            Discharge 2026-06-01 21:44:00  Discharge Coordinator   

                location acuity_level  patient_age patient_gender  \
1132      Reception Desk  ESI Level 2  

In [53]:
# Preview the output
display(df_events.head(15))

,case_id,activity,timestamp,resource,location,acuity_level,patient_age,patient_gender,attending_doctor,paperwork_missing_flag
0,1000,Check-in,2026-06-01 21:59:00,Front Desk Staff,Reception Desk,ESI Level 3,32,Male,Dr. Smith,True
1,1000,Triage,2026-06-01 22:06:00,Nurse Sarah,Triage Room 1,ESI Level 3,32,Male,Dr. Smith,True
2,1000,Doctor Consultation,2026-06-01 22:58:00,Dr. Smith,Consultation Bay B,ESI Level 3,32,Male,Dr. Smith,True
3,1000,X-Ray,2026-06-01 23:21:00,Tech Mike,Radiology Lab 2,ESI Level 3,32,Male,Dr. Smith,True
4,1000,Triage,2026-06-01 23:37:00,Nurse Sarah,Triage Room 1,ESI Level 3,32,Male,Dr. Smith,True
5,1000,Doctor Consultation,2026-06-01 23:47:00,Dr. Smith,Consultation Bay B,ESI Level 3,32,Male,Dr. Smith,False
6,1000,Discharge,2026-06-02 00:04:00,Discharge Coordinator,Exit Lobby,ESI Level 3,32,Male,Dr. Smith,True
7,1001,Check-in,2026-06-03 17:16:00,Front Desk Staff,Reception Desk,ESI Level 2,45,Male,Dr. Chen,False
8,1001,Triage,2026-06-03 17:28:00,Nurse Sarah,Triage Room 1,ESI Level 2,45,Male,Dr. Chen,False
9,1001,Doctor Consultation,2026-06-03 18:11:00,Dr. Chen,Consultation Bay B,ESI Level 2,45,Male,Dr. Chen,False


In [55]:
# Save to CSV for Task 1 BigQuery ingestion
df_events.to_csv("careflow_synthetic_ehr_logs.csv", index=False)
print(
    "\nDataset generated successfully! Saved as 'careflow_synthetic_ehr_logs.csv'."
)


Dataset generated successfully! Saved as 'careflow_synthetic_ehr_logs.csv'.


Data Cleaning Phase


In [56]:
# 1. Load the raw dataset
df = pd.read_csv("careflow_synthetic_ehr_logs.csv")

print(f"Original row count: {len(df)}")

Original row count: 1142


In [57]:
# 2. Drop rows with missing critical process columns
df = df.dropna(subset=["case_id", "activity", "timestamp"])

In [58]:
# 3. Convert timestamp to actual datetime objects
df["timestamp"] = pd.to_datetime(df["timestamp"])

In [59]:
# 4. Standardize text strings (strip whitespace, capitalize properly)
df["activity"] = df["activity"].str.strip()
df["resource"] = df["resource"].str.strip()
df["location"] = df["location"].str.strip()

In [60]:
# 5. Remove exact duplicate events
df = df.drop_duplicates()

In [61]:
# 6. Sort chronologically per case ID
df = df.sort_values(by=["case_id", "timestamp"]).reset_index(drop=True)

In [64]:
# 7. Save the cleaned dataset
df.to_csv("careflow_cleaned_ehr_logs.csv", index=False)
print(f"Cleaned row count: {len(df)}")
print("Dataset cleaned successfully and saved as 'careflow_cleaned_ehr_logs.csv'!")

Cleaned row count: 1142
Dataset cleaned successfully and saved as 'careflow_cleaned_ehr_logs.csv'!


In [65]:
# Preview the output
display(df_events.head(15))

,case_id,activity,timestamp,resource,location,acuity_level,patient_age,patient_gender,attending_doctor,paperwork_missing_flag
0,1000,Check-in,2026-06-01 21:59:00,Front Desk Staff,Reception Desk,ESI Level 3,32,Male,Dr. Smith,True
1,1000,Triage,2026-06-01 22:06:00,Nurse Sarah,Triage Room 1,ESI Level 3,32,Male,Dr. Smith,True
2,1000,Doctor Consultation,2026-06-01 22:58:00,Dr. Smith,Consultation Bay B,ESI Level 3,32,Male,Dr. Smith,True
3,1000,X-Ray,2026-06-01 23:21:00,Tech Mike,Radiology Lab 2,ESI Level 3,32,Male,Dr. Smith,True
4,1000,Triage,2026-06-01 23:37:00,Nurse Sarah,Triage Room 1,ESI Level 3,32,Male,Dr. Smith,True
5,1000,Doctor Consultation,2026-06-01 23:47:00,Dr. Smith,Consultation Bay B,ESI Level 3,32,Male,Dr. Smith,False
6,1000,Discharge,2026-06-02 00:04:00,Discharge Coordinator,Exit Lobby,ESI Level 3,32,Male,Dr. Smith,True
7,1001,Check-in,2026-06-03 17:16:00,Front Desk Staff,Reception Desk,ESI Level 2,45,Male,Dr. Chen,False
8,1001,Triage,2026-06-03 17:28:00,Nurse Sarah,Triage Room 1,ESI Level 2,45,Male,Dr. Chen,False
9,1001,Doctor Consultation,2026-06-03 18:11:00,Dr. Chen,Consultation Bay B,ESI Level 2,45,Male,Dr. Chen,False


Dataset Description

In [66]:
df.shape

(1142, 10)

Attributes of the dataset

In [67]:
df.columns

Index(['case_id', 'activity', 'timestamp', 'resource', 'location',
       'acuity_level', 'patient_age', 'patient_gender', 'attending_doctor',
       'paperwork_missing_flag'],
      dtype='object')

Overview of the Dataset

In [68]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1142 entries, 0 to 1141
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   case_id                 1142 non-null   int64         
 1   activity                1142 non-null   object        
 2   timestamp               1142 non-null   datetime64[ns]
 3   resource                1142 non-null   object        
 4   location                1142 non-null   object        
 5   acuity_level            1142 non-null   object        
 6   patient_age             1142 non-null   int64         
 7   patient_gender          1142 non-null   object        
 8   attending_doctor        1142 non-null   object        
 9   paperwork_missing_flag  1142 non-null   bool          
dtypes: bool(1), datetime64[ns](1), int64(2), object(6)
memory usage: 81.5+ KB


Summary Statistics of the Numerical Features

In [69]:
df.describe()

,case_id,timestamp,patient_age
count,1142.000000,1142,1142.000000
mean,1099.119089,2026-06-02 20:32:31.208406016,54.056918
min,1000.000000,2026-06-01 08:58:00,18.000000
25%,1047.250000,2026-06-02 04:38:00,37.000000
50%,1099.000000,2026-06-02 18:58:30,56.000000
75%,1150.000000,2026-06-03 13:00:45,72.000000
max,1199.000000,2026-06-04 09:55:00,85.000000
std,58.401912,NaN,19.220737
